In [15]:
import os
import sys
from dotenv import load_dotenv

if "google.colab" in sys.modules:
    # Running in Colab
    from google.colab import drive
    drive.mount("/content/drive")

    BASE_FOLDER = "/content/drive/Shareddrives"
    BASE_FOLDER_GIT = "/content"

    from google.colab import userdata
    DB_HOST = userdata.get("SUPABASE_DB_HOST")
    DB_PORT = userdata.get("SUPABASE_DB_PORT")
    DB_NAME = userdata.get("SUPABASE_DB_NAME")
    DB_USER = userdata.get("SUPABASE_DB_USER")
    DB_PASSWORD = userdata.get("SUPABASE_DB_PASSWORD")

else:
    # Running in local Jupyter
    BASE_FOLDER = r"G:\Shared drives"
    BASE_FOLDER_GIT = r"C:\Users\Windows 11\Notebook"

    load_dotenv()
    DB_HOST = os.getenv("SUPABASE_DB_HOST")
    DB_PORT = os.getenv("SUPABASE_DB_PORT")
    DB_NAME = os.getenv("SUPABASE_DB_NAME")
    DB_USER = os.getenv("SUPABASE_DB_USER")
    DB_PASSWORD = os.getenv("SUPABASE_DB_PASSWORD")

print("Using folder:", BASE_FOLDER)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using folder: /content/drive/Shareddrives


In [43]:
import pandas as pd


def filter_last_1y_from_latest(
    df: pd.DataFrame,
    date_col: str = "BILLDATE",
) -> pd.DataFrame:
    """
    Keep rows within 1 year from the latest date in date_col.
    Invalid dates are ignored.

    Returns filtered dataframe (copy).
    """

    d = pd.to_datetime(df[date_col], errors="coerce")

    # remove rows with invalid date
    valid_mask = d.notna()
    df_valid = df.loc[valid_mask].copy()
    d_valid = d.loc[valid_mask]

    latest = d_valid.max()
    cutoff = latest - pd.DateOffset(years=1)

    df_out = df_valid.loc[d_valid >= cutoff].copy()

    return df_out

In [16]:
raw_folder = f"{BASE_FOLDER}/KCW-Data/kcw_analytics/01_raw"
curated_folder = f"{BASE_FOLDER}/KCW-Data/kcw_analytics/03_curated"

In [37]:
df_hq_icmas = pd.read_csv(f"{raw_folder}/raw_hq_icmas_products.csv", dtype="string")
df_hq_pidet = pd.read_csv(f"{raw_folder}/raw_hq_pidet_purchase_lines.csv", dtype="string")
df_fact_sales_all = pd.read_csv(f"{curated_folder}/fact_sales_all.csv", dtype="string")

In [44]:
df_hq_icmas = df_hq_icmas[df_hq_icmas["STATUS"] == "1"].copy()

In [45]:
df_hq_pidet = filter_last_1y_from_latest(df_hq_pidet, "BILLDATE")
df_fact_sales_all = filter_last_1y_from_latest(df_fact_sales_all, "BILLDATE")

In [28]:
from io import StringIO
import csv
import pandas as pd


def refresh_table_via_staging_df(
    conn,
    df: pd.DataFrame,
    main_table: str,
    staging_table: str,
    source_file: str | None = None,
) -> dict:
    """
    Load DataFrame into staging table, validate, then replace main table from staging.

    Parameters
    ----------
    conn :
        psycopg connection
    df : pd.DataFrame
        DataFrame to load
    main_table : str
        Target main table name, e.g. 'raw_kcw.raw_hq_icmas_products'
    staging_table : str
        Staging table name, e.g. 'raw_kcw.raw_hq_icmas_products_stg'
    source_file : str | None
        Optional source file name to stamp into _source_file if that column exists

    Returns
    -------
    dict
    """
    if df is None or df.empty:
        raise ValueError("Input DataFrame is empty. Main table not touched.")

    load_df = df.copy()

    # keep column order from df
    cols = load_df.columns.tolist()
    quoted_cols = ", ".join([f'"{c}"' for c in cols])

    # convert pandas NA/NaN to empty CSV fields so COPY reads them as NULL/empty
    load_df = load_df.astype("object").where(pd.notna(load_df), None)

    buffer = StringIO()
    load_df.to_csv(
        buffer,
        index=False,
        header=True,
        encoding="utf-8",
        quoting=csv.QUOTE_MINIMAL,
    )
    buffer.seek(0)

    copy_sql = f"""
    COPY {staging_table} ({quoted_cols})
    FROM STDIN WITH CSV HEADER
    """

    with conn.cursor() as cur:
        # ensure staging exists
        cur.execute(f"""
        create table if not exists {staging_table}
        (like {main_table} including all)
        """)

        # clear staging only
        cur.execute(f"delete from {staging_table}")

        # load staging from dataframe CSV buffer
        with cur.copy(copy_sql) as copy:
            while data := buffer.read(1024 * 1024):
                copy.write(data)

        # stamp source file only if requested and column exists
        if source_file is not None:
            cur.execute(
                """
                select 1
                from information_schema.columns
                where table_schema = %s
                  and table_name = %s
                  and column_name = '_source_file'
                """,
                tuple(staging_table.split(".", 1)) if "." in staging_table else ("public", staging_table),
            )
            has_source_file = cur.fetchone() is not None

            if has_source_file:
                cur.execute(
                    f"""
                    update {staging_table}
                    set _source_file = %s
                    where _source_file is null
                    """,
                    (source_file,),
                )

        # validate staging count
        cur.execute(f"select count(*) from {staging_table}")
        staging_count = cur.fetchone()[0]

        if staging_count == 0:
            raise ValueError("Staging load produced 0 rows. Main table not touched.")

        # replace main from staging
        cur.execute(f"delete from {main_table}")
        cur.execute(f"""
            insert into {main_table}
            select * from {staging_table}
        """)

        cur.execute(f"select count(*) from {main_table}")
        main_count = cur.fetchone()[0]

    conn.commit()

    return {
        "status": "ok",
        "staging_rows": staging_count,
        "main_rows": main_count,
        "source_file": source_file,
        "main_table": main_table,
        "staging_table": staging_table,
    }

In [29]:
!pip install psycopg[binary]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.8/212.8 kB 13.7 MB/s eta 0:00:00


In [32]:
import psycopg

def get_conn():
    return psycopg.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
    )

conn = get_conn()

In [51]:
result = refresh_table_via_staging_df(
    conn=conn,
    df=df_hq_icmas,
    main_table="raw_kcw.raw_hq_icmas_products",
    staging_table="raw_kcw.raw_hq_icmas_products_stg",
)

In [49]:
result = refresh_table_via_staging_df(
    conn=conn,
    df=df_hq_pidet,
    main_table="raw_kcw.raw_hq_pidet_purchase_lines",
    staging_table="raw_kcw.raw_hq_pidet_purchase_lines_stg",
)

In [50]:
result = refresh_table_via_staging_df(
    conn=conn,
    df=df_fact_sales_all,
    main_table="raw_kcw.fact_sales_all",
    staging_table="raw_kcw.fact_sales_all_stg",
)